In [4]:
from jupyter_dash import JupyterDash
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from animal_shelter import AnimalShelter

In [5]:
username = "aacuser"
password = "SNHU1234"

db = AnimalShelter(username, password)

df = pd.DataFrame.from_records(db.read({}))

if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

In [6]:
WATER_RESCUE_BREEDS = [
    "Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"
]

MOUNTAIN_WILDERNESS_BREEDS = [
    "German Shepherd", "Alaskan Malamute", "Old English Sheepdog",
    "Siberian Husky", "Rottweiler"
]

DISASTER_TRACKING_BREEDS = [
    "Doberman Pinscher", "German Shepherd", "Golden Retriever",
    "Bloodhound", "Rottweiler"
]


def water_rescue_query():
    return {
        "breed": {"$in": WATER_RESCUE_BREEDS},
        "sex_upon_outcome": "Intact Female",
        "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
    }


def mountain_wilderness_query():
    return {
        "breed": {"$in": MOUNTAIN_WILDERNESS_BREEDS},
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
    }


def disaster_tracking_query():
    return {
        "breed": {"$in": DISASTER_TRACKING_BREEDS},
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
    }

In [7]:
app = JupyterDash(__name__)

image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Center([
        html.A(
            html.Img(
                src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style={'height': '150px'}
            ),
            href='https://www.snhu.edu', target='_blank'
        )
    ]),
    html.Center(html.B(html.H1('Grazioso Salvare Dashboard'))),
    html.Center(html.P('Developed by: Tanvir Mahmod')),
    html.Hr(),
    html.Div(
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'Disaster'},
                {'label': 'Reset', 'value': 'Reset'},
            ],
            value='Reset',
            labelStyle={'display': 'inline-block', 'margin-right': '20px'}
        ),
        style={'textAlign': 'center'}
    ),
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        editable=False,
        filter_action='native',
        sort_action='native',
        sort_mode='multi',
        column_selectable='single',
        row_selectable='single',
        row_deletable=False,
        selected_columns=[],
        selected_rows=[0],
        page_action='native',
        page_current=0,
        page_size=10,
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left', 'minWidth': '100px'},
    ),
    html.Br(),
    html.Hr(),
    html.Div(className='row',
             style={'display': 'flex'},
             children=[
                 html.Div(id='graph-id', className='col s12 m6'),
                 html.Div(id='map-id', className='col s12 m6')
             ])
])

In [8]:
@app.callback(Output('datatable-id', 'data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    if filter_type == 'Water':
        query_df = pd.DataFrame.from_records(db.read(water_rescue_query()))
    elif filter_type == 'Mountain':
        query_df = pd.DataFrame.from_records(db.read(mountain_wilderness_query()))
    elif filter_type == 'Disaster':
        query_df = pd.DataFrame.from_records(db.read(disaster_tracking_query()))
    else:
        query_df = pd.DataFrame.from_records(db.read({}))

    if '_id' in query_df.columns:
        query_df.drop(columns=['_id'], inplace=True)

    return query_df.to_dict('records')


@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    if viewData is None:
        return []
    dff = pd.DataFrame.from_dict(viewData)
    if dff.empty:
        return [html.P("No data available for this filter.")]

    return [
        dcc.Graph(
            figure=px.pie(dff, names='breed', title='Preferred Animals by Breed')
        )
    ]


@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    if viewData is None:
        return
    dff = pd.DataFrame.from_dict(viewData)
    if dff.empty:
        return

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75, -97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[dff.loc[row, 'location_lat'], dff.loc[row, 'location_long']], children=[
                dl.Tooltip(dff.loc[row, 'breed']),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.loc[row, 'name'])
                ])
            ])
        ])
    ]


app.run_server()

Dash app running on https://monkeydynamic-regardcourage-3000.codio.io/proxy/8050/
